In [89]:
pip install transformers

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip



                                              0.0/9.1 MB ? eta -:--:--
                                              0.1/9.1 MB 1.7 MB/s eta 0:00:06
                                              0.2/9.1 MB 2.8 MB/s eta 0:00:04
     -                                        0.3/9.1 MB 3.5 MB/s eta 0:00:03
     -                                        0.3/9.1 MB 3.5 MB/s eta 0:00:03
     ---                                      0.9/9.1 MB 4.9 MB/s eta 0:00:02
     ----                                     1.1/9.1 MB 5.3 MB/s eta 0:00:02
     -----                                    1.2/9.1 MB 4.9 MB/s eta 0:00:02
     -----                                    1.3/9.1 MB 4.9 MB/s eta 0:00:02
     -------                                  1.7/9.1 MB 5.3 MB/s eta 0:00:02
     --------                                 1.9/9.1 MB 5.2 MB/s eta 0:00:02
     ---------                                2.1/9.1 MB 5.3 MB/s eta 0:00:02
     ---------                                2.3/9.1 MB 5.1 MB/s eta 

In [103]:
#imports
import spacy
import pdfplumber
from docx import Document
import re
import pandas
import pytesseract
from PIL import Image
import docx
import fitz  # PyMuPDF
from transformers import BertTokenizer, BertModel
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np



In [91]:
# Load NLP model
nlp = spacy.load('en_core_web_sm')

def extract_text_from_pdf(file_path):
    with pdfplumber.open(file_path) as pdf:
        text = ''
        for page in pdf.pages:
            text += page.extract_text()
    return text


In [92]:
def extract_text_from_docx(file_path):
    doc = Document(file_path)
    text = '\n'.join([para.text for para in doc.paragraphs])
    print("Text extraction from DOCX complete.")
    return text


In [93]:
def extract_text_with_font_size(docx_file):
    doc = docx.Document(docx_file)
    text_with_font_size = []

    for para in doc.paragraphs:
        for run in para.runs:
            if run.font.size:
                text_with_font_size.append((run.text, run.font.size.pt))
    
    return text_with_font_size

In [94]:
def extract_largest_text(text_with_font_size):
    if not text_with_font_size:
        return None
    
    largest_text = max(text_with_font_size, key=lambda x: x[1])
    return largest_text[0]

In [95]:

def extract_email(text):
    email_pattern = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b')
    matches = email_pattern.findall(text)
    print(f"Email extraction complete: {matches[0] if matches else 'No email found'}")
    return matches[0] if matches else ''    

In [96]:
def extract_phone(text):
    # Improved phone number pattern to capture various formats
    phone_pattern = re.compile(r'\+?\d[\d -]{8,}\d')
    matches = phone_pattern.findall(text)
    print(f"Phone number extraction complete: {matches[0] if matches else 'No phone number found'}")
    return matches[0] if matches else ''


In [97]:
def extract_name(text):
    # Split the text by lines and iterate through each line
    for line in text.split('\n'):
        # Check if the line contains typical name format (First Name Last Name)
        if line.count(' ') >= 1:
            return tuple(line.strip().split(' ', 1))  # Split the line into first name and last name
    return None, None  # Return None if name format is not found

In [98]:
def parse_cv(file_path, file_type):
    if file_type == 'pdf':
        text = extract_text_from_pdf(file_path)
    elif file_type == 'docx':
        text_with_font_size = extract_text_with_font_size(file_path)
        largest_text = extract_largest_text(text_with_font_size)
        text = largest_text if largest_text else ""
    else:
        raise ValueError('Unsupported file type')
    
    email = extract_email(text)
    phone = extract_phone(text)
    name = extract_name(text)
    
    extracted_info = {
        'name': name,
        'email': email,
        'phone': phone,
        'education': [],
        'experience': [],
        'skills': [],
        'address': [],
        'certifications': [],
        'projects': []
    }
    
    sections = text.split('\n')
    current_section = None
    section_keywords = {
        'education': ['education', 'coursework'],
        'experience': ['experience', 'employment', 'work'],
        'skills': ['skills', 'technologies'],
        'address': ['address', 'location'],
        'certifications': ['certifications', 'awards', 'achievements'],
        'projects': ['projects', 'portfolio']
    }

    for line in sections:
        line_lower = line.lower()
        if any(keyword in line_lower for keyword in section_keywords['education']):
            current_section = 'education'
            continue
        elif any(keyword in line_lower for keyword in section_keywords['experience']):
            current_section = 'experience'
            continue
        elif any(keyword in line_lower for keyword in section_keywords['skills']):
            current_section = 'skills'
            continue
        elif any(keyword in line_lower for keyword in section_keywords['address']):
            current_section = 'address'
            continue
        elif any(keyword in line_lower for keyword in section_keywords['certifications']):
            current_section = 'certifications'
            continue
        elif any(keyword in line_lower for keyword in section_keywords['projects']):
            current_section = 'projects'
            continue

        if current_section and line.strip():
            extracted_info[current_section].append(line.strip())

    # Split skills into individual items
    if 'skills' in extracted_info:
        skill_list = []
        for skill in extracted_info['skills']:
            skill_list.extend([s.strip() for s in re.split(r'[,:;]', skill)])
        extracted_info['skills'] = skill_list

    return extracted_info

In [99]:
# Example usage for parsing the CV
if __name__ == "__main__":
    # Specify the file path and file type
    file_path = 'D://Projects//SkillStudio//ai_model//MY_CV (3).pdf'
    file_type = 'pdf'
    
    # Parse the CV
    parsed_cv = parse_cv(file_path, file_type)
    
    # Print the extracted information
    print(parsed_cv)

Email extraction complete: priyanshurouth@gmail.com
Phone number extraction complete: +91 9875403824
{'name': ('Priyanshu', 'Routh'), 'email': 'priyanshurouth@gmail.com', 'phone': '+91 9875403824', 'education': ['University of Engineering and Management 2021 - Present', 'B.Tech(CSIT) Current GPA: 8.4/10', 'Saint Luke’s Day School 2005 - 2021', 'ICSE (2019) & ISC (2021) 79% & 85.07%', 'Courses: Object-Oriented Programming, Data Structures & Algorithms, Cloud Computing & IoT, Database', 'Management Systems, Deep Learning, Operating Systems, Artificial Intelligence & Machine Learning, Blockchain', 'Technology, Software Engineering.'], 'experience': ['IBM Skill Build Micro Internship March 2024 - April 2024', 'Data Science Intern', '• Included Aptitude tests, coding challenges, and AI-based Personal Interview System.'], 'skills': ['Languages', 'C', 'Kotlin', 'Python', 'Java', 'JavaScript/TypeScript', 'HTML/CSS', 'SQL', 'C++', 'C', 'Tools', 'Git/GitHub', 'Unix Shell', 'VS Code', 'IntelliJ',

In [100]:
def score_cv(extracted_info, job_requirements):
    score = 0
    max_score = 0
    improvements = {}

    # Skills scoring
    skills = set(extracted_info.get('skills', []))
    required_skills = set(job_requirements.get('skills', []))
    matching_skills = skills.intersection(required_skills)
    score += len(matching_skills)
    max_score += len(required_skills)
    if not matching_skills:
        improvements['skills'] = f"Add skills: {', '.join(required_skills - skills)}"

    # Education scoring
    education = ' '.join(extracted_info.get('education', [])).lower()
    required_education = job_requirements.get('education', [])
    if any(edu.lower() in education for edu in required_education):
        score += 1
    else:
        improvements['education'] = f"Consider adding educational qualifications like: {', '.join(required_education)}"
    max_score += 1

    # Experience scoring
    experience = ' '.join(extracted_info.get('experience', [])).lower()
    required_experience = job_requirements.get('experience', [])
    if any(exp.lower() in experience for exp in required_experience):
        score += 1
    else:
        improvements['experience'] = f"Consider adding experience like: {', '.join(required_experience)}"
    max_score += 1

    # Calculate percentage score
    ats_score = (score / max_score) * 100

    return ats_score, improvements

In [101]:
# Define Job Requirements
job_requirements = {
    'skills': ['GO', 'aws', 'Rust', 'Augmented Reality', 'Data Science'],
    'education': ['B.Tech', 'M.Tech', 'B.Sc', 'M.Sc'],
    'experience': ['Intern', 'Junior', 'Senior']
}

In [102]:
# Example usage
if __name__ == "__main__":
    file_path = 'D://Projects//SkillStudio//ai_model//MY_CV (3).pdf'
    file_type = 'pdf'
    parsed_cv = parse_cv(file_path, file_type)
    print(parsed_cv)

    ats_score, improvements = score_cv(parsed_cv, job_requirements)
    print(f"ATS Score: {ats_score}%")
    print("Areas of Improvement:", improvements)

Email extraction complete: priyanshurouth@gmail.com
Phone number extraction complete: +91 9875403824
{'name': ('Priyanshu', 'Routh'), 'email': 'priyanshurouth@gmail.com', 'phone': '+91 9875403824', 'education': ['University of Engineering and Management 2021 - Present', 'B.Tech(CSIT) Current GPA: 8.4/10', 'Saint Luke’s Day School 2005 - 2021', 'ICSE (2019) & ISC (2021) 79% & 85.07%', 'Courses: Object-Oriented Programming, Data Structures & Algorithms, Cloud Computing & IoT, Database', 'Management Systems, Deep Learning, Operating Systems, Artificial Intelligence & Machine Learning, Blockchain', 'Technology, Software Engineering.'], 'experience': ['IBM Skill Build Micro Internship March 2024 - April 2024', 'Data Science Intern', '• Included Aptitude tests, coding challenges, and AI-based Personal Interview System.'], 'skills': ['Languages', 'C', 'Kotlin', 'Python', 'Java', 'JavaScript/TypeScript', 'HTML/CSS', 'SQL', 'C++', 'C', 'Tools', 'Git/GitHub', 'Unix Shell', 'VS Code', 'IntelliJ',